# Test version compatibility

Covers all logical branches of `_confirm_version` and its helper functions, across different server configurations.

In [1]:
from firefly_client import FireflyClient

# only needed for the mock scenario
from firefly_client._server_compat import is_server_compatible
from unittest.mock import patch, MagicMock

In [2]:
# Uncomment for debugging outputs
FireflyClient._debug = True

In [3]:
def pprint_confirm_version(result):
    """For pretty-printing the result of _confirm_version() for debugging."""
    print(f"reachable:      {result['reachable']}")
    print(f"compatible:     {result['compatible']}")
    print(f"server_version: {result['server_version']!r}")
    try:
        raw = result['response'].json()
    except Exception:
        raw = result['response'].text[:200]
    print(f"raw response:   {raw}")

## Compatible server
Creating a FireflyClient instance should succeed without warnings or errors.

### Base Firefly app

In [4]:
fc_base = FireflyClient.make_client(url='http://localhost:8080/firefly', launch_browser=False)

DEBUG: new instance: http://localhost:8080/firefly


`_confirm_version()` should return `reachable=True` and `compatible=True` and note that the server version is under the `Version` key.

In [5]:
pprint_confirm_version(fc_base._confirm_version())

reachable:      True
compatible:     True
server_version: '2026.1-DEV:FIREFLY-1331-version-validation_c5e3'
raw response:   {'Version': '2026.1-DEV:FIREFLY-1331-version-validation_c5e3', 'Built On': 'Tue Apr 07 14:25:15 PDT 2026', 'Git commit': 'c5e3756bf'}


### App using Firefly

In [6]:
fc_app = FireflyClient.make_client(url='http://localhost:8080/irsaviewer/', launch_browser=False)

DEBUG: new instance: http://localhost:8080/irsaviewer/


`_confirm_version()` should return `reachable=True` and `compatible=True` and note that the version appears under the `Firefly Library Version` key instead.

In [7]:
pprint_confirm_version(fc_app._confirm_version())

reachable:      True
compatible:     True
server_version: '2026.1-DEV:FIREFLY-1331-version-validation_c5e3'
raw response:   {'Firefly Git Commit': 'c5e3756bf', 'Version': 'v1.0', 'Built On': 'Tue Apr 07 14:25:55 PDT 2026', 'Git commit': '1331d2dd', 'Firefly Library Version': '2026.1-DEV:FIREFLY-1331-version-validation_c5e3'}


## Incompatible server version

Intercept the version endpoint response of the local irsaviewer and substitute an old version string, then verify that creating a FireflyClient instance raises `ValueError` with the message how to rectify the issue.

We don't have such a server available yet so we'll just mock the response of `_confirm_version` to simulate this scenario.

In [8]:
INCOMPATIBLE_VERSION = '2025.5.5'  # below the MIN_SERVER_VERSION (2025.6-DEV)

mock_resp_incompat = MagicMock()
mock_resp_incompat.status_code = 200
mock_resp_incompat.json.return_value = {'Version': INCOMPATIBLE_VERSION}

ver_incompat = {
    'reachable': True,
    'compatible': is_server_compatible(INCOMPATIBLE_VERSION),
    'server_version': INCOMPATIBLE_VERSION,
    'response': mock_resp_incompat,
}

In [9]:
# Patch _confirm_version at class level so the mock takes effect inside make_client()
with patch.object(FireflyClient, '_confirm_version', return_value=ver_incompat):
    FireflyClient.make_client(url='http://localhost:8080/firefly/', launch_browser=False)

ValueError: Version of the provided Firefly server http://localhost:8080/firefly/ is not compatible with this version of firefly_client.
  Server version: 2025.5.5
  Required: >=2026.1-DEV
  Please use the URL of a compatible Firefly server


`_confirm_version()` should return `reachable=True` and `compatible=False`.

In [10]:
pprint_confirm_version(ver_incompat)

reachable:      True
compatible:     False
server_version: '2025.5.5'
raw response:   {'Version': '2025.5.5'}


## Server's Version endpoint not reachable

When the version endpoint returns a non-200 response, creating a FireflyClient instance should emit a warning and proceed rather than raise an error.

In [11]:
fc_no_version = FireflyClient.make_client(url='https://irsa.ipac.caltech.edu/irsaviewer/', launch_browser=False)

DEBUG: Firefly server's version endpoint response: {'success': False, 'error': {}}
DEBUG: new instance: https://irsa.ipac.caltech.edu/irsaviewer/


`_confirm_version()` should return `reachable=False` and hence `compatible=False` and `server_version=None`.

In [12]:
pprint_confirm_version(fc_no_version._confirm_version())

reachable:      False
compatible:     False
server_version: None
raw response:   {'success': False, 'error': {}}
